In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("processed_data.csv")

In [3]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['id', 'date']).reset_index(drop=True)

appcat_cols = [c for c in df.columns if c.startswith('appCat')]

In [4]:
df['valence_x_arousal'] = df['circumplex.valence'] * df['circumplex.arousal']

In [5]:
df['activity_x_screen'] = df['activity'] / (df['screen'] + 1e-6)

In [6]:
df['total_app_usage'] = df[appcat_cols].sum(axis=1)

In [7]:
df['social_ratio'] = df['appCat.social'] / (df['total_app_usage'] + 1e-6)

In [8]:
for col in ['mood', 'screen', 'total_app_usage']:
    df[f'{col}_z'] = df.groupby('id')[col].transform(
        lambda x: (x - x.mean()) / x.std()
    )

In [9]:
rolling_cols_behavior = ['activity', 'screen_z', 'total_app_usage_z',
                         'appCat.social', 'appCat.communication',
                         'appCat.entertainment']
rolling_cols_mood     = ['mood_z', 'circumplex.valence', 'circumplex.arousal']
rolling_std_cols      = ['mood_z', 'circumplex.valence']
lag1_cols = ['circumplex.valence', 'circumplex.arousal', 'activity',
             'screen_z', 'valence_x_arousal', 'activity_x_screen',
             'total_app_usage_z', 'social_ratio']

In [10]:
WINDOW = 3
all_instances = []

In [11]:
for pid, group in df.groupby('id'):
    group = group.sort_values('date').reset_index(drop=True)
    feat = pd.DataFrame()
    feat['id']   = group['id']
    feat['date'] = group['date']
    feat['mood_target'] = group['mood']

    for lag in range(1, WINDOW + 1):
        feat[f'mood_lag{lag}'] = group['mood_z'].shift(lag)

    for col in lag1_cols:
        feat[f'{col}_lag1'] = group[col].shift(1)

    for col in rolling_cols_behavior + rolling_cols_mood:
        feat[f'{col}_rmean3'] = group[col].shift(1).rolling(WINDOW).mean()

    for col in rolling_std_cols:
        feat[f'{col}_rstd3'] = group[col].shift(1).rolling(WINDOW).std()

    feat['mood_trend'] = group['mood_z'].shift(1) - group['mood_z'].shift(2)

    feat['valence_trend'] = group['circumplex.valence'].shift(1) - group['circumplex.valence'].shift(2)

    all_instances.append(feat)

In [12]:
instance_df = pd.concat(all_instances, ignore_index=True)

In [13]:
df

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,mood,screen,sms,valence_x_arousal,activity_x_screen,total_app_usage,social_ratio,mood_z,screen_z,total_app_usage_z
0,AS14.01,2014-03-21,0.134050,2.725836,4.027056,3.989428,1.973108,0.0,4.067350,3.432983,...,6.20,153.665872,0.0,0.040000,0.000872,31.996180,0.135410,-1.508946,1.197044,1.482122
1,AS14.01,2014-03-22,0.236880,2.172608,3.964560,3.864134,2.082603,0.0,0.000000,2.709411,...,6.40,47.613651,1.0,0.300000,0.004975,25.760900,0.140832,-1.172378,-1.430057,-0.031481
2,AS14.01,2014-03-23,0.142741,2.575234,3.997136,3.479659,2.472391,0.0,0.000000,2.575445,...,6.80,67.067644,0.0,0.160000,0.002128,25.915725,0.151751,-0.499242,-0.948148,0.006103
3,AS14.01,2014-03-24,0.078961,1.788840,3.882423,4.813171,1.899766,0.0,1.388791,2.492627,...,6.00,76.772199,0.0,0.000000,0.001029,29.541991,0.146582,-1.845514,-0.707749,0.886372
4,AS14.01,2014-03-25,0.098374,2.400335,4.254726,3.558287,2.465002,0.0,0.000000,3.427617,...,6.75,169.097283,1.0,0.250000,0.000582,26.910580,0.157636,-0.583384,1.579307,0.247602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1209,AS14.33,2014-05-26,0.115201,2.164623,3.564463,4.246735,0.000000,0.0,4.879235,2.881751,...,5.40,300.399810,1.0,-0.000000,0.000383,22.455718,0.210143,-1.361717,1.650060,0.271533
1210,AS14.33,2014-05-27,0.012704,2.256099,4.100707,4.148628,0.000000,0.0,0.000000,2.935757,...,6.20,113.607750,1.0,-0.240000,0.000112,20.359936,0.193365,-0.453485,-0.753582,-0.390668
1211,AS14.33,2014-05-28,0.103301,2.359948,3.643849,3.804606,0.000000,0.0,3.822943,3.363561,...,8.20,113.657683,1.0,0.000000,0.000909,23.412981,0.196813,1.817094,-0.752940,0.573998
1212,AS14.33,2014-05-29,0.169354,2.008504,4.014365,4.239440,0.000000,0.0,0.000000,2.373416,...,7.00,118.978033,1.0,-1.333333,0.001423,19.550935,0.270970,0.454746,-0.684477,-0.646287


In [14]:
instance_df

,id,date,mood_target,mood_lag1,mood_lag2,mood_lag3,circumplex.valence_lag1,circumplex.arousal_lag1,activity_lag1,screen_z_lag1,...,appCat.social_rmean3,appCat.communication_rmean3,appCat.entertainment_rmean3,mood_z_rmean3,circumplex.valence_rmean3,circumplex.arousal_rmean3,mood_z_rstd3,circumplex.valence_rstd3,mood_trend,valence_trend
0,AS14.01,2014-03-21,6.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AS14.01,2014-03-22,6.40,-1.508946,NaN,NaN,0.200000,0.200000,0.134050,1.197044,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AS14.01,2014-03-23,6.80,-1.172378,-1.508946,NaN,0.500000,0.600000,0.236880,-1.430057,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.336568,0.300000
3,AS14.01,2014-03-24,6.00,-0.499242,-1.172378,-1.508946,0.800000,0.200000,0.142741,-0.948148,...,3.964432,3.996251,3.777741,-1.060189,0.500000,0.333333,0.514116,0.300000,0.673136,0.300000
4,AS14.01,2014-03-25,6.75,-1.845514,-0.499242,-1.172378,0.000000,0.800000,0.078961,-0.707749,...,3.963671,3.948040,4.052321,-1.172378,0.433333,0.533333,0.673136,0.404145,-1.346272,-0.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1209,AS14.33,2014-05-26,5.40,-0.396721,-0.226427,-0.869758,0.333333,-0.750000,0.161470,-1.165675,...,4.602289,3.660810,4.714670,-0.497635,0.194444,-0.777778,0.333326,0.173472,-0.170293,0.083333
1210,AS14.33,2014-05-27,6.20,-1.361717,-0.396721,-0.226427,0.000000,-0.400000,0.115201,1.650060,...,4.415564,3.634228,4.278555,-0.661621,0.194444,-0.633333,0.612250,0.173472,-0.964996,-0.333333
1211,AS14.33,2014-05-28,8.20,-0.453485,-1.361717,-0.396721,0.400000,-0.600000,0.012704,-0.753582,...,4.165333,3.747984,4.193683,-0.737307,0.244444,-0.583333,0.541499,0.214303,0.908232,0.400000
1212,AS14.33,2014-05-29,7.00,1.817094,-0.453485,-1.361717,1.200000,0.000000,0.103301,-0.752940,...,4.421263,3.769673,4.066656,0.000631,0.533333,-0.333333,1.637338,0.611010,2.270579,0.800000


In [15]:
instance_df = instance_df.dropna().reset_index(drop=True)

In [16]:
feature_cols = [c for c in instance_df.columns if c not in ['id', 'date', 'mood_target']]

In [17]:
print(f"Final dataset shape:{instance_df.shape}")

Final dataset shape:(1133, 27)


In [18]:
print(f"Number of features:{len(feature_cols)}")

Number of features:24


In [19]:
print(f"\nFeature list:\n{feature_cols}")


Feature list:
['mood_lag1', 'mood_lag2', 'mood_lag3', 'circumplex.valence_lag1', 'circumplex.arousal_lag1', 'activity_lag1', 'screen_z_lag1', 'valence_x_arousal_lag1', 'activity_x_screen_lag1', 'total_app_usage_z_lag1', 'social_ratio_lag1', 'activity_rmean3', 'screen_z_rmean3', 'total_app_usage_z_rmean3', 'appCat.social_rmean3', 'appCat.communication_rmean3', 'appCat.entertainment_rmean3', 'mood_z_rmean3', 'circumplex.valence_rmean3', 'circumplex.arousal_rmean3', 'mood_z_rstd3', 'circumplex.valence_rstd3', 'mood_trend', 'valence_trend']


In [20]:
pd.set_option('display.max_columns', None)

In [21]:
instance_df.head(10)

,id,date,mood_target,mood_lag1,mood_lag2,mood_lag3,circumplex.valence_lag1,circumplex.arousal_lag1,activity_lag1,screen_z_lag1,valence_x_arousal_lag1,activity_x_screen_lag1,total_app_usage_z_lag1,social_ratio_lag1,activity_rmean3,screen_z_rmean3,total_app_usage_z_rmean3,appCat.social_rmean3,appCat.communication_rmean3,appCat.entertainment_rmean3,mood_z_rmean3,circumplex.valence_rmean3,circumplex.arousal_rmean3,mood_z_rstd3,circumplex.valence_rstd3,mood_trend,valence_trend
0,AS14.01,2014-03-24,6.00,-0.499242,-1.172378,-1.508946,0.80,0.2,0.142741,-0.948148,0.160,0.002128,0.006103,0.151751,0.171224,-0.393720,0.485581,3.964432,3.996251,3.777741,-1.060189,0.500000,0.333333,0.514116,0.300000,0.673136,0.30
1,AS14.01,2014-03-25,6.75,-1.845514,-0.499242,-1.172378,0.00,0.8,0.078961,-0.707749,0.000,0.001029,0.886372,0.146582,0.152861,-1.028651,0.286998,3.963671,3.948040,4.052321,-1.172378,0.433333,0.533333,0.673136,0.404145,-1.346272,-0.80
2,AS14.01,2014-03-26,6.60,-0.583384,-1.845514,-0.499242,0.50,0.5,0.098374,1.579307,0.250,0.000582,0.247602,0.157636,0.106692,-0.025530,0.380026,4.168378,4.044762,3.950372,-0.976047,0.433333,0.500000,0.754155,0.404145,1.262130,0.50
3,AS14.01,2014-03-27,7.00,-0.835810,-0.583384,-1.845514,0.60,-0.2,0.101308,-0.271333,-0.120,0.001073,-0.341523,0.197241,0.092881,0.200075,0.264150,4.467195,4.003210,4.415726,-1.088236,0.366667,0.366667,0.667856,0.321455,-0.252426,0.10
4,AS14.01,2014-03-28,6.40,-0.162675,-0.835810,-0.583384,0.80,0.2,0.159511,0.591008,0.160,0.001235,0.629158,0.150613,0.119731,0.632994,0.178412,4.453693,3.973131,4.237065,-0.527290,0.633333,0.166667,0.340056,0.152753,0.673136,0.20
5,AS14.01,2014-03-29,8.00,-1.172378,-0.162675,-0.835810,0.60,-0.6,0.095698,-1.223583,-0.360,0.001710,1.974579,0.096468,0.118839,-0.301302,0.754072,4.133767,3.628895,4.559680,-0.723621,0.666667,-0.200000,0.514116,0.115470,-1.009704,-0.20
6,AS14.01,2014-03-30,7.50,1.520165,-1.172378,-0.162675,1.00,0.2,0.068203,0.611813,0.200,0.000524,-0.147787,0.162022,0.107804,-0.006920,0.818650,3.889434,3.633509,4.499733,0.061704,0.800000,-0.066667,1.360223,0.200000,2.692544,0.40
7,AS14.01,2014-03-31,7.40,0.678745,1.520165,-1.172378,0.75,-0.5,0.049093,1.540395,-0.375,0.000293,-0.143694,0.159740,0.070998,0.309542,0.561033,3.806568,3.749611,4.789119,0.342177,0.783333,-0.300000,1.377464,0.202073,-0.841420,-0.25
8,AS14.01,2014-04-01,6.00,0.510461,0.678745,1.520165,0.60,0.0,0.070505,-0.244860,0.000,0.000739,1.856139,0.120700,0.062600,0.635783,0.521553,4.061768,3.924168,4.602156,0.903124,0.783333,-0.100000,0.540957,0.202073,-0.168284,-0.15
9,AS14.01,2014-04-02,6.50,-1.845514,0.510461,0.678745,0.00,0.2,0.118443,-0.370437,0.000,0.001310,2.024194,0.112368,0.079347,0.308366,1.245547,3.978465,3.805432,4.297828,-0.218769,0.450000,-0.100000,1.411313,0.396863,-2.355976,-0.60


In [22]:
cols = ['mood_lag1', 'mood_lag2', 'mood_lag3', 'circumplex.valence_lag1', 
        'circumplex.arousal_lag1', 'activity_lag1', 'screen_z_lag1', 
        'valence_x_arousal_lag1', 'activity_x_screen_lag1', 'total_app_usage_z_lag1', 
        'social_ratio_lag1', 'activity_rmean3', 'screen_z_rmean3', 
        'total_app_usage_z_rmean3', 'appCat.social_rmean3', 'appCat.communication_rmean3', 
        'appCat.entertainment_rmean3', 'mood_z_rmean3', 'circumplex.valence_rmean3', 
        'circumplex.arousal_rmean3', 'mood_z_rstd3', 'circumplex.valence_rstd3', 
        'mood_trend', 'valence_trend']
range_df = instance_df[cols].agg(['min', 'max']).T.reset_index()
range_df.columns = ['variable', 'min', 'max']
print(range_df)

                       variable       min       max
0                     mood_lag1 -4.094698  3.020473
1                     mood_lag2 -4.094698  3.020473
2                     mood_lag3 -4.094698  3.369977
3       circumplex.valence_lag1 -1.250000  2.000000
4       circumplex.arousal_lag1 -2.000000  2.000000
5                 activity_lag1  0.000000  0.549430
6                 screen_z_lag1 -2.063615  3.052254
7        valence_x_arousal_lag1 -2.880000  2.240000
8        activity_x_screen_lag1  0.000000  0.034200
9        total_app_usage_z_lag1 -3.812582  3.648021
10            social_ratio_lag1  0.000000  1.000000
11              activity_rmean3  0.002632  0.457917
12              screen_z_rmean3 -1.362843  2.079838
13     total_app_usage_z_rmean3 -1.992690  1.875861
14         appCat.social_rmean3  0.000000  6.169980
15  appCat.communication_rmean3  1.070401  5.559571
16  appCat.entertainment_rmean3  0.000000  6.190046
17                mood_z_rmean3 -3.195900  1.764231
18    circum

In [23]:
instance_df.to_csv("ML_Dataset.csv", index=False)